# PAPER REVIEW

___
# General Learning and Techniques

### Deep Learning (2016)
*Y. Bengio, I. Goodfellow and A. Courville, Deep learning, MIT Press, 2016,
http://www.deeplearningbook.org.*

___
# Deep Hedging

### Deep Hedging (2019)
*Buehler, H., Gonon, L., Teichmann, J. and Wood, B., Deep hedging.
Quant. Finance, 2019, 19, 1271–1291.*

- Deep hedging uses a feedforward neural network to efficiently hedge a european option

### Equal risk pricing of derivatives with deep hedging (2020)
*Carbonneau, A., Godin, F.*

### Efficient analytic approximation of the optimal hedging strategy for a European call option with transaction costs (2006)
*Zakamouline, V.I.*

- Multiple methods to add transaction cost analytically (or semi) to the Black-Scholes delta hedging framework

### Deep Hedging: Learning to Remove the Drift under Trading Frictions with Minimal Equivalent Near-Martingale Measures (2022)
*Hans Buehler, Phillip Murray, Mikko S. Pakkanen, and Ben Wood. Deep hedging: Learning to remove the drift under trading frictions with minimal equivalent nearmartingale measures. https://arxiv.org/abs/2111.07844, 2022.*

- In efficient market with no transaction cost, we can define a matringale measure from utility function (convex, etc.)
- Cannot make arbitrage in perfect market implies no statistical arbitrage under Q in trading cost market.
- General result: Considering that any equivalent true martingale measure is also a near-martingale measure, this result is a formalization of the intuitive notion that in order to avoid statistical arbitrage we do not truly have to find a full martingale measure, but that we only have to "bend" the drifts of our trading instruments enough to be dominated by prevailing trading cost.


___
# Volatility Modeling

### SANOS: Smooth strictly Arbitrage-free Non-parametric Option Surfaces (2026)
*Hans Buehler, Blanka Horvath, Anastasis Kratsios, Yannick Limmer, Raeid Saqur*

**Overview**
- Create a smooth and stricly arbitrage-free non-parametric option surface (by construction, not constraint)
- Linear complexity (efficient)
- Require only positive $\Sigma_t$ for arbitrage free


**Research Orientation**
- introduce a dyamic volatility surface simulator. Use the paper "Deep Hedging: Learning to Remove the Drift," Risk, Feb 2022, to remove the drift and be able to efficiently simulate the volatility surface, with no calendar spread
1. Implement SANOS's static LP/DLV fitting on real SPX data
2. Fit SANOS day-by-day over a historical window and extract time series of $\Sigma_t$ and empirically characterize it (statistics, autcorrelation, cross-sectional dependance in strike)
3. Train a model (AR(1), GARCH), then train a generative model (autoencoder to Neural SDE, or VAE), and verify arbitrage (calendar and dynamic)
4. Potentially remove drift

Notes: Autoencoder to Neural SDE: *Since $\Sigma_j^i \geq 0$ is the only constraint SANOS's DLV parametrization needs, design the autoencoder's decoder output layer with a positivity-enforcing activation (softplus, exp, or square), then every decoded output — regardless of what the latent code $z_t$ or the Neural SDE does upstream — is automatically a valid, smooth, arbitrage-free SANOS surface. Static arbitrage-freedom for free, by construction*

- Multi-asset modeling

Explaining SANOS:
- SANOS optimizes the density weights (over the whole extended strike grid) so that the resulting price matches known market prices at the quoted strikes; between strikes, the price/density is smoothed by using Black-Scholes call kernels (i.e. each grid point is treated as a mini lognormal diffusion) with a fixed vol — scaled off ATM vol — purely for smoothing, not fit to market.
- SANOS represents the pure call price at strike k as a weighted sum of smoothed Black-Scholes kernels centered at every model strike i, where the weights form a fitted martingale (risk-neutral) density: the Breeden-Litzenberger density
$$
C(k) = \sum_{i} q_i \, C_{BS}\!\left(\frac{k}{K_i}\sigma_i\sqrt{T}\right) K_i, 
$$
where $\qquad \sum_i q_i = 1,\quad \sum_i q_i K_i = 1,\quad q_i \geq 0$

Here, the number/range of strike points are extended (more i) (so that gaps between market strikes is not too large). 
- The newly inserted strikes do get a value via linear interpolation (the become i's in the above formula)

What to read:
- discrete local volatility
- martingale density of call options (Breeden-Litzenberger: ∂²C/∂K²)

**Steps for SANOS**
1. Market data prep — pure (forward/discount-normalized) strikes, bid/ask, sqrtT per expiry, with sanity checks (bid< ask, bid ≥intrinsic, ATM variance increasing across expiries) — ExpiryData.

2. Build the extended strike grid xstrikes: insert evenly-spaced (np.linspace) points so no gap exceeds max_dx inside the quoted range, plus extrapolate out to min_k/max_k in the tails. This only decides where the mixture components live.
- Extra points inserted between market strikes to not exceed max_dx
- Extrapolation points beyond the market range to put density beyond min/max (to integrate to 1)

3. Assign each grid point a kernel vol xvols_i — a fixed input, not fitted. By default (vol_mode="atm") it's flat: every point gets the same ATM vol. This only sets kernel width, not the density.

- SANOS fits the weights xdensity_i (only) so that C(k) = Σ_i xdensity_i · BS_call(k/xstrikes_i, √T·xvols_i) · xstrikes_i matches the market-known call prices at the quoted strikes. xstrikes_i (the grid) and xvols_i (flat, 25% of ATM vol by default) are fixed inputs, not optimized; xvols_i only smooths the curve between grid points and has no relation to market prices.

4. One convex optimization solves the density AND the smile fit simultaneously. The solver fits xdensity_i (=q_i) directly as the LP/QP variable, subject to it being a valid martingale measure (q_i≥0, Σq_i=1, Σq_iK_i=1), while at the same time minimizing the price error to market mid/bid-ask. Since C(k) is already a direct, smooth function of the density (the mixture sum), no differentiation of a fitted smile is needed.

- The objective, in words: choose xdensity_i to minimize the (weighted L1 or L2) distance between the model's price C(k), evaluated only at the market-quoted strikes
- K_i (location) — where the bump is centered. Fixed by the grid.
- q_i (mass) — how much total probability the bump carries. This is the only thing being optimized.
- v_i (width)

5. The calendar no-arbitrage constraint, enforced inside that same optimization, not as a later pass: expiry j's fitted price curve must dominate expiry j-1's everywhere. This can be done sequentially (method="iterative", solve expiry by expiry, each floored by the previous) or, more rigorously, jointly across all expiries in one single big convex program (method="global", the default and theoretically cleanest option.

6. (Optional, after fitting) interpolate between the fitted expiries in time via the monotone cubic spline over ATM variance, for pricing at expiries that weren't directly quoted.

### Discrete Local Volatility for Large Time Steps (2015-2020)
*Hans Buehler, Evgeny Ryskin*

### CVI: Convex Volatility Interpolation (2026)
*Deschatres, F.*

- Use convex optimization to fit a volatility surface
- No-arbitrage are constraint (contrary to SANOS, which are built-in)
- Surface is fitted in the variance space (it is a choice, can be price space as well), this means no-calendar arbitrage is a linear constraint, no-butterfly-arbitrage conditions are linear in the wings but become nonlinear for non-extreme strikes
- Parametrization with arbitrary number of parameters
- The variance $v(K)$ should be represented using a set of basis functions, specifically cubic B-splines, and is assumed to be a linear function of $\text{log}(K)$ for extreme strikes.

___
# Generative AI


### Arbitrage-free neural-SDE market models (2021)
*Samuel N. Cohen Christoph Reisinger Sheng Wang*

### Estimating risks of option books using neural-SDE market models (2022)
*Samuel N. Cohen Christoph Reisinger Sheng Wang*

### Multi-asset spot and option market simulation (2021)
*Magnus Wiese, Ben Wood, Alexandre Pachoud, Ralf Korn, Hans Buehler, Phillip Murray, and Lianjun Bai. Multi-asset spot and option market simulation. https: //arxiv.org/abs/2112.06823, 2021.*

### Modeling Asset Price Process: An Approach for Imaging Price Chart with Generative Diffusion Models (2024)
*Jinseong Park · Hyungjin Ko · Jaewook Lee*

___
# Stochastic Models

### Interest Rate Model (2006)
*Brigo, D., Mercurio, F.*

- Main book on interest rate stochastic model
- Used for the G2++ model simulation

### A General Gaussian Interest Rate Model Consistent with the Current Term Structure (2012)
*Francesco, M. D.*

- Gn++ model characteristics

### Pricing Swaptions and Coupon Bond Options in Affine Term Structure Models (2012)
*Schrager, D.F., Pelsser, A.A.J*

- Swaption semi-analytical approximation for the Gn++ model
- Approximation come from fixing some variables to their martingale value, this approximate $\text{E}[X^2]$ by $(\text{E}[X])^2$, second order approximation. This approximation work well for near ATM swaptions, and not so well for deep ITM or OTM swations.
- This method is used to calibrate a Gn++ model to observed market ATM swaption prices.

### Efficient Simulation of the Heston Stochastic Volatility Model (2007)
*Andersen, L.*

- Heston model properties